In [1]:
from ortools.algorithms.python import knapsack_solver
import numpy as np
import platform
import time

In [2]:
NUMBER_OF_TRIALS = 100

import problems

# Every implementation in this project solves the identical set of problems by
# reading them from problems.py.
problem_set = problems.generate()

num_problems = problem_set.num_problems
num_items_per_problem = problem_set.num_items
max_capacity = problem_set.max_capacity
capacities = problem_set.capacities

items = problem_set.items

print(f"{num_problems} problems, {num_items_per_problem} items each, capacity at most {max_capacity}")
for i in range(3):
    row = ", ".join(str(item) for item in problem_set.items[i])
    print(f"Problem {i + 1}: capacity {problem_set.capacities[i]}, items [{row}]")

10000 problems, 16 items each, capacity at most 2000
Problem 1: capacity 1954, items [90, 774, 654, 439, 433, 858, 86, 697, 202, 95, 526, 975, 736, 761, 717, 786]
Problem 2: capacity 1296, items [513, 128, 839, 450, 500, 371, 183, 926, 781, 644, 403, 822, 545, 443, 451, 228]
Problem 3: capacity 62, items [93, 555, 888, 64, 858, 827, 277, 632, 166, 758, 700, 355, 68, 970, 446, 893]


In [3]:
# How much the answers discriminate: a problem whose items can reach its capacity
# exactly has an answer anything could guess, one that falls short does not.
problem_set.analyze()

4962/10000 problems (49.6%) have a subset that sums to the capacity exactly; the rest fall short of it


0.4962

In [4]:
# OR-Tools Section

print(platform.node())

# Function to solve a single subset sum problem using OR-Tools
def solve_subset_sum(items, capacity, problem_idx, totals, chosen):
    solver = knapsack_solver.KnapsackSolver(
        knapsack_solver.KNAPSACK_DYNAMIC_PROGRAMMING_SOLVER, 'SubsetSumExample')

    # Subset sum is knapsack with an item's value equal to its weight, so
    # OR-Tools is given the same array for both.
    solver.init(items, [items], [capacity])

    totals[problem_idx] = solver.solve()
    for i in range(len(items)):
        chosen[problem_idx, i] = solver.best_solution_contains(i)

# Storage for the totals and for which items made up each one
ortools_totals = np.zeros(num_problems, dtype=np.int32)
ortools_chosen = np.zeros((num_problems, num_items_per_problem), dtype=np.uint8)

execution_times = []
for _ in range(NUMBER_OF_TRIALS):
    # Measure execution time
    start_time = time.time()

    for i in range(num_problems):
        solve_subset_sum(items[i], capacities[i], i, ortools_totals, ortools_chosen)

    # Print execution time
    end_time = time.time()
    execution_times.append(end_time - start_time)

print(f"Average CPU execution time: {(sum(execution_times) / NUMBER_OF_TRIALS):.6f} seconds")

# Each reported total has to be the sum of the items it names, and the best available
print(f"{problems.check_solutions(problem_set, ortools_totals, ortools_chosen)} bad solutions")

# Print the final results
#for i in range(num_problems):
for i in range(10):
    print(f"Problem {i + 1}: total {ortools_totals[i]} "
          f"from items {np.flatnonzero(ortools_chosen[i]).tolist()}")

Andrews-MacBook-Air.local
Average CPU execution time: 0.138790 seconds
0 bad solutions
Problem 1: total 1954 from items [1, 2, 10]
Problem 2: total 1296 from items [3, 10, 13]
Problem 3: total 0 from items []
Problem 4: total 239 from items [3, 7]
Problem 5: total 1349 from items [1, 4, 5, 11]
Problem 6: total 1303 from items [1, 6, 8]
Problem 7: total 725 from items [0, 5, 7, 14]
Problem 8: total 1291 from items [1, 4, 5, 10]
Problem 9: total 717 from items [9, 14, 15]
Problem 10: total 939 from items [10, 14]


In [5]:
# Bitset DP Section
#
# The same recurrence the GPU kernels run, but with the reachable totals packed one
# per bit of a single integer instead of one per byte of a table, so a shift and an
# OR advance the whole table by one item. problems.best_subset keeps the reachable
# set after each item so it can walk backwards and recover the items themselves.
print(platform.node())

bitset_totals = np.zeros(num_problems, dtype=np.int32)
bitset_chosen = np.zeros((num_problems, num_items_per_problem), dtype=np.uint8)

execution_times = []
for _ in range(NUMBER_OF_TRIALS):
    # Measure execution time
    start_time = time.time()

    for i in range(num_problems):
        total, picked = problems.best_subset(items[i], int(capacities[i]))
        bitset_totals[i] = total
        bitset_chosen[i] = 0
        bitset_chosen[i, picked] = 1

    # Print execution time
    end_time = time.time()
    execution_times.append(end_time - start_time)

print(f"Average bitset DP execution time: {(sum(execution_times) / NUMBER_OF_TRIALS):.6f} seconds")

# Each reported total has to be the sum of the items it names, and the best available
print(f"{problems.check_solutions(problem_set, bitset_totals, bitset_chosen)} bad solutions")

# Print the results
#for i in range(num_problems):
for i in range(10):
    print(f"Problem {i + 1}: total {bitset_totals[i]} "
          f"from items {np.flatnonzero(bitset_chosen[i]).tolist()}")

Andrews-MacBook-Air.local
Average bitset DP execution time: 0.051305 seconds
0 bad solutions
Problem 1: total 1954 from items [1, 2, 10]
Problem 2: total 1296 from items [3, 10, 13]
Problem 3: total 0 from items []
Problem 4: total 239 from items [3, 7]
Problem 5: total 1349 from items [1, 4, 5, 11]
Problem 6: total 1303 from items [1, 6, 8]
Problem 7: total 725 from items [0, 5, 7, 14]
Problem 8: total 1291 from items [1, 4, 5, 10]
Problem 9: total 717 from items [9, 14, 15]
Problem 10: total 939 from items [10, 14]


In [6]:
# Metal Section

# This section should run on the M4 MacBook Air
import Metal

# Metal kernel to solve multiple subset sum problems. One threadgroup solves one
# problem; its threads split the table between them.
#
# The reachable totals are packed one per bit, exactly as problems.best_subset packs
# them into a Python integer, so a shift and an OR advance the whole table by one item
# instead of touching one byte per total. The set after each item is kept, which is
# what lets the kernel walk backwards afterwards and recover the items themselves.
kernel_code = """
#include <metal_stdlib>
using namespace metal;

kernel void subset_sum(device const int *items [[buffer(0)]],
                       device const int *capacities [[buffer(1)]],
                       device int *max_values [[buffer(2)]],
                       device uchar *chosen [[buffer(3)]],
                       constant int &num_items [[buffer(4)]],
                       constant int &max_capacity [[buffer(5)]],
                       threadgroup uint *history [[threadgroup(0)]],
                       uint problem_idx [[threadgroup_position_in_grid]],
                       uint thread_idx [[thread_position_in_threadgroup]],
                       uint threads_per_group [[threads_per_threadgroup]]) {
    int words = (max_capacity + 32) / 32;

    // history + i * words is the set reachable using only the first i items.
    // Before any item, the only reachable total is zero.
    for (int j = thread_idx; j < words; j += threads_per_group) {
        history[j] = (j == 0) ? 1u : 0u;
    }
    threadgroup_barrier(mem_flags::mem_threadgroup);

    device const int *problem_items = items + problem_idx * num_items;
    for (int i = 0; i < num_items; i++) {
        int item = problem_items[i];
        int word_shift = item >> 5;   // whole words the bits move up by
        int bit_shift = item & 31;    // and the leftover bits within a word
        threadgroup uint *current = history + i * words;
        threadgroup uint *next = current + words;

        for (int j = thread_idx; j < words; j += threads_per_group) {
            uint shifted = 0;
            int high = j - word_shift;
            if (high >= 0) {
                shifted = current[high] << bit_shift;
                // A shift that is not a whole number of words also pulls in the top
                // bits of the word below. Shifting a uint by 32 is undefined, so a
                // bit_shift of zero has to skip that part.
                if (bit_shift != 0 && high >= 1) {
                    shifted |= current[high - 1] >> (32 - bit_shift);
                }
            }
            next[j] = current[j] | shifted;
        }
        threadgroup_barrier(mem_flags::mem_threadgroup);
    }

    // One thread reads off the answer and walks the history back to find the items
    if (thread_idx == 0) {
        threadgroup uint *reachable = history + num_items * words;

        int w = capacities[problem_idx];
        while (((reachable[w >> 5] >> (w & 31)) & 1u) == 0u) {
            w--;
        }
        max_values[problem_idx] = w;

        // If the total was already reachable without item i then item i was not
        // needed; otherwise it was, so subtract it and carry on down.
        device uchar *problem_chosen = chosen + problem_idx * num_items;
        int remaining = w;
        for (int i = num_items - 1; i >= 0; i--) {
            threadgroup uint *without = history + i * words;
            if (((without[remaining >> 5] >> (remaining & 31)) & 1u) != 0u) {
                problem_chosen[i] = 0;
            } else {
                problem_chosen[i] = 1;
                remaining -= problem_items[i];
            }
        }
    }
}
"""

device = Metal.MTLCreateSystemDefaultDevice()
print(device.name())

# Compile the kernel code
library, error = device.newLibraryWithSource_options_error_(kernel_code, None, None)
if library is None:
    raise RuntimeError(f"Metal kernel failed to compile: {error}")
pipeline, error = device.newComputePipelineStateWithFunction_error_(
    library.newFunctionWithName_("subset_sum"), None)
if pipeline is None:
    raise RuntimeError(f"Metal pipeline could not be created: {error}")
command_queue = device.newCommandQueue()

# One snapshot of the packed table per item, plus the starting one. Metal wants the
# threadgroup allocation to be a multiple of 16 bytes.
words = (max_capacity + 32) // 32
table_bytes = (num_items_per_problem + 1) * words * 4
threadgroup_memory_size = (table_bytes + 15) // 16 * 16

# One thread per word of the table measured fastest on the M4: the strided loop then
# runs exactly once per thread, and the barrier after each item stays narrow. Filling
# the threadgroup instead makes every barrier wait on threads with nothing to do.
threads_per_threadgroup = min(words, pipeline.maxTotalThreadsPerThreadgroup())

# Allocate the buffers once, outside the timed loop. Apple silicon shares one pool
# of memory between the CPU and GPU, so numpy writes straight into the memory the
# kernel reads and there is no host-to-device transfer to make at all.
items_gpu = device.newBufferWithLength_options_(
    items.nbytes, Metal.MTLResourceStorageModeShared)
capacities_gpu = device.newBufferWithLength_options_(
    capacities.nbytes, Metal.MTLResourceStorageModeShared)
max_values_gpu = device.newBufferWithLength_options_(
    num_problems * 4, Metal.MTLResourceStorageModeShared)
chosen_gpu = device.newBufferWithLength_options_(
    num_problems * num_items_per_problem, Metal.MTLResourceStorageModeShared)

np.frombuffer(items_gpu.contents().as_buffer(items.nbytes),
              dtype=np.int32).reshape(items.shape)[:] = items
np.frombuffer(capacities_gpu.contents().as_buffer(capacities.nbytes),
              dtype=np.int32)[:] = capacities

# Views on the output buffers, so reading the results is not a copy either
metal_totals = np.frombuffer(
    max_values_gpu.contents().as_buffer(num_problems * 4), dtype=np.int32)
metal_chosen = np.frombuffer(
    chosen_gpu.contents().as_buffer(num_problems * num_items_per_problem),
    dtype=np.uint8).reshape(num_problems, num_items_per_problem)

num_items_bytes = np.int32(num_items_per_problem).tobytes()
max_capacity_bytes = np.int32(max_capacity).tobytes()

execution_times = []
gpu_times = []
for _ in range(NUMBER_OF_TRIALS):
    # Measure execution time
    start_time = time.time()

    # Launch the kernel
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline)
    encoder.setBuffer_offset_atIndex_(items_gpu, 0, 0)
    encoder.setBuffer_offset_atIndex_(capacities_gpu, 0, 1)
    encoder.setBuffer_offset_atIndex_(max_values_gpu, 0, 2)
    encoder.setBuffer_offset_atIndex_(chosen_gpu, 0, 3)
    encoder.setBytes_length_atIndex_(num_items_bytes, 4, 4)
    encoder.setBytes_length_atIndex_(max_capacity_bytes, 4, 5)
    encoder.setThreadgroupMemoryLength_atIndex_(threadgroup_memory_size, 0)
    encoder.dispatchThreadgroups_threadsPerThreadgroup_(
        Metal.MTLSizeMake(num_problems, 1, 1), Metal.MTLSizeMake(threads_per_threadgroup, 1, 1))
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    if command_buffer.error() is not None:
        raise RuntimeError(f"Metal kernel failed: {command_buffer.error()}")

    # Print execution time
    end_time = time.time()
    execution_times.append(end_time - start_time)
    gpu_times.append(command_buffer.GPUEndTime() - command_buffer.GPUStartTime())

print(f"Average Metal execution time: {(sum(execution_times) / NUMBER_OF_TRIALS):.6f} seconds")
print(f"Average Metal kernel time (GPU only): {(sum(gpu_times) / NUMBER_OF_TRIALS):.6f} seconds")

# Each reported total has to be the sum of the items it names, and the best available
print(f"{problems.check_solutions(problem_set, metal_totals, metal_chosen)} bad solutions")

# Print the results
#for i in range(num_problems):
for i in range(10):
    print(f"Problem {i + 1}: total {metal_totals[i]} "
          f"from items {np.flatnonzero(metal_chosen[i]).tolist()}")

Apple M4
Average Metal execution time: 0.001879 seconds
Average Metal kernel time (GPU only): 0.001502 seconds
0 bad solutions
Problem 1: total 1954 from items [1, 2, 10]
Problem 2: total 1296 from items [3, 10, 13]
Problem 3: total 0 from items []
Problem 4: total 239 from items [3, 7]
Problem 5: total 1349 from items [1, 4, 5, 11]
Problem 6: total 1303 from items [1, 6, 8]
Problem 7: total 725 from items [0, 5, 7, 14]
Problem 8: total 1291 from items [1, 4, 5, 10]
Problem 9: total 717 from items [9, 14, 15]
Problem 10: total 939 from items [10, 14]
